### IDS

In [6]:
import time

def is_safe(state, row, col):
    for r in range(row):
        c = state[r]
        if c == col or abs(c - col) == abs(r - row):
            return False
    return True

def dfs(state, row, n, limit):
    if row == limit:
        return state if row == n else None

    for col in range(n):
        if is_safe(state, row, col):
            state[row] = col
            result = dfs(state.copy(), row + 1, n, limit)
            if result:
                return result
    return None

def iterative_deepening_search(n):
    for depth in range(1, n + 1):
        result = dfs([-1] * n, 0, n, depth)
        if result:
            return result
    return None

def board_to_string(solution):
    n = len(solution)
    lines = []
    for row in range(n):
        line = ""
        for col in range(n):
            line += "Q " if solution[row] == col else ". "
        lines.append(line)
    return "\n".join(lines) + "\n"

def run_nqueens_experiment(n, trials=10):
    output_lines = []
    output_lines.append(f"Running {trials} trials for {n}-Queens problem...\n")
    total_time = 0
    final_solution = None

    for i in range(trials):
        start = time.time()
        solution = iterative_deepening_search(n)
        elapsed = time.time() - start
        total_time += elapsed
        output_lines.append(f"Trial {i+1}: Time = {elapsed:.4f} seconds")
        if solution:
            board_str = board_to_string(solution)
            output_lines.append("Solution:\n" + board_str)
            final_solution = solution
        else:
            output_lines.append("No solution found.\n")

    avg_time = total_time / trials
    output_lines.append(f"===== {n}-Queens Summary =====")
    output_lines.append("Final solution:")
    if final_solution:
        output_lines.append(board_to_string(final_solution))
    else:
        output_lines.append("No solution found.")
    output_lines.append(f"Average time over {trials} trials: {avg_time:.4f} seconds\n")

    filename = f"{n}-Queens__IDS_result.txt"
    with open(filename, "w") as f:
        f.write("\n".join(output_lines))
    print("\n".join(output_lines))

In [7]:
run_nqueens_experiment(8, trials=10)

Running 10 trials for 8-Queens problem...

Trial 1: Time = 0.0169 seconds
Solution:
Q . . . . . . . 
. . . . Q . . . 
. . . . . . . Q 
. . . . . Q . . 
. . Q . . . . . 
. . . . . . Q . 
. Q . . . . . . 
. . . Q . . . . 

Trial 2: Time = 0.0154 seconds
Solution:
Q . . . . . . . 
. . . . Q . . . 
. . . . . . . Q 
. . . . . Q . . 
. . Q . . . . . 
. . . . . . Q . 
. Q . . . . . . 
. . . Q . . . . 

Trial 3: Time = 0.0149 seconds
Solution:
Q . . . . . . . 
. . . . Q . . . 
. . . . . . . Q 
. . . . . Q . . 
. . Q . . . . . 
. . . . . . Q . 
. Q . . . . . . 
. . . Q . . . . 

Trial 4: Time = 0.0140 seconds
Solution:
Q . . . . . . . 
. . . . Q . . . 
. . . . . . . Q 
. . . . . Q . . 
. . Q . . . . . 
. . . . . . Q . 
. Q . . . . . . 
. . . Q . . . . 

Trial 5: Time = 0.0141 seconds
Solution:
Q . . . . . . . 
. . . . Q . . . 
. . . . . . . Q 
. . . . . Q . . 
. . Q . . . . . 
. . . . . . Q . 
. Q . . . . . . 
. . . Q . . . . 

Trial 6: Time = 0.0143 seconds
Solution:
Q . . . . . . . 
. . . . Q

In [8]:
# run_nqueens_experiment(50, trials=10) # 跑不動QQ

### HC Algorithm

In [9]:
import time
import random


def count_attacks_fast(state):
    n = len(state)
    col_counts = [0] * n
    diag1 = [0] * (2 * n - 1)  # row - col + n - 1
    diag2 = [0] * (2 * n - 1)  # row + col

    for row in range(n):
        col = state[row]
        col_counts[col] += 1
        diag1[row - col + n - 1] += 1
        diag2[row + col] += 1

    def count_pairs(counts):
        return sum(c * (c - 1) // 2 for c in counts if c > 1)

    return count_pairs(col_counts) + count_pairs(diag1) + count_pairs(diag2)


def get_random_neighbors(state, sample_size=50):
    n = len(state)
    neighbors = []
    for _ in range(sample_size):
        row = random.randint(0, n - 1)
        new_col = random.choice([c for c in range(n) if c != state[row]])
        new_state = state.copy()
        new_state[row] = new_col
        neighbors.append(new_state)
    return neighbors


def hill_climbing_once(n, neighbor_sample=50):
    current = [random.randint(0, n - 1) for _ in range(n)]
    best_attacks = count_attacks_fast(current)

    while True:
        neighbors = get_random_neighbors(current, sample_size=neighbor_sample)
        better_neighbors = [s for s in neighbors if count_attacks_fast(s) < best_attacks]

        if better_neighbors:
            current = min(better_neighbors, key=count_attacks_fast)
            best_attacks = count_attacks_fast(current)
        else:
            break

    return current, best_attacks


def hill_climbing(n, max_restarts=100, neighbor_sample=50):
    best_overall = None
    best_attacks = float("inf")

    for _ in range(max_restarts):
        solution, attacks = hill_climbing_once(n, neighbor_sample)
        if attacks == 0:
            return solution, 0
        if attacks < best_attacks:
            best_overall = solution
            best_attacks = attacks

    return best_overall, best_attacks

def board_to_string(state):
    if state is None:
        return "No solution found.\n"
    n = len(state)
    lines = []
    for row in range(n):
        line = ""
        for col in range(n):
            line += "Q " if state[row] == col else ". "
        lines.append(line)
    return "\n".join(lines) + "\n"

def run_experiment(n, trials, max_restarts=100, neighbor_sample=50):
    total_time = 0
    success_count = 0
    output_lines = []
    output_lines.append(f"Running {trials} trials for {n}-Queens using Hill Climbing (sample size={neighbor_sample}) with {max_restarts} restarts...\n")

    for trial in range(trials):
        start_time = time.time()
        solution, attacks = hill_climbing(n, max_restarts, neighbor_sample)
        elapsed_time = time.time() - start_time
        total_time += elapsed_time

        output_lines.append(f"Trial {trial + 1}:")
        output_lines.append(f"  Solution found? {'Yes' if attacks == 0 else 'No'}")
        output_lines.append(f"  Number of attacking pairs: {attacks}")
        output_lines.append(f"  Execution time: {elapsed_time:.4f} seconds")
        output_lines.append("  Final board state:")
        if n <= 20:
            output_lines.append(board_to_string(solution))
        else:
            output_lines.append("  [Board output skipped due to size]")
        output_lines.append("=" * 40)

        if attacks == 0:
            success_count += 1

    avg_time = total_time / trials
    success_rate = success_count / trials

    output_lines.append("===== Final Results =====")
    output_lines.append(f"Average time over {trials} trials: {avg_time:.4f} seconds")
    output_lines.append(f"Success rate (0 attacking pairs): {success_rate:.2%}")
    output_lines.append(f"Number of successes (0 attacking pairs): {success_count}")

    filename = f"{n}-Queens_HC_result.txt"
    with open(filename, "w") as f:
        f.write("\n".join(output_lines))

    print("\n".join(output_lines))


In [10]:
run_experiment(8, trials=30, max_restarts=100)

Running 30 trials for 8-Queens using Hill Climbing (sample size=50) with 100 restarts...

Trial 1:
  Solution found? Yes
  Number of attacking pairs: 0
  Execution time: 0.0432 seconds
  Final board state:
. . . . . . . Q 
. Q . . . . . . 
. . . . Q . . . 
. . Q . . . . . 
Q . . . . . . . 
. . . . . . Q . 
. . . Q . . . . 
. . . . . Q . . 

Trial 2:
  Solution found? Yes
  Number of attacking pairs: 0
  Execution time: 0.0212 seconds
  Final board state:
. . . . . Q . . 
. . Q . . . . . 
. . . . Q . . . 
. . . . . . . Q 
Q . . . . . . . 
. . . Q . . . . 
. Q . . . . . . 
. . . . . . Q . 

Trial 3:
  Solution found? Yes
  Number of attacking pairs: 0
  Execution time: 0.0203 seconds
  Final board state:
. . . . . . . Q 
. . . Q . . . . 
Q . . . . . . . 
. . Q . . . . . 
. . . . . Q . . 
. Q . . . . . . 
. . . . . . Q . 
. . . . Q . . . 

Trial 4:
  Solution found? Yes
  Number of attacking pairs: 0
  Execution time: 0.0140 seconds
  Final board state:
. . Q . . . . . 
. . . . Q . . . 
.

In [11]:
run_experiment(50, trials=30, max_restarts=1000)

Running 30 trials for 50-Queens using Hill Climbing (sample size=50) with 1000 restarts...

Trial 1:
  Solution found? No
  Number of attacking pairs: 8
  Execution time: 23.9784 seconds
  Final board state:
  [Board output skipped due to size]
Trial 2:
  Solution found? No
  Number of attacking pairs: 6
  Execution time: 24.1130 seconds
  Final board state:
  [Board output skipped due to size]
Trial 3:
  Solution found? No
  Number of attacking pairs: 8
  Execution time: 23.5627 seconds
  Final board state:
  [Board output skipped due to size]
Trial 4:
  Solution found? No
  Number of attacking pairs: 6
  Execution time: 23.6298 seconds
  Final board state:
  [Board output skipped due to size]
Trial 5:
  Solution found? No
  Number of attacking pairs: 8
  Execution time: 24.1595 seconds
  Final board state:
  [Board output skipped due to size]
Trial 6:
  Solution found? No
  Number of attacking pairs: 8
  Execution time: 24.3700 seconds
  Final board state:
  [Board output skipped due

### GA algorithm

In [12]:
import random
import time

# Count the number of attacking queen pairs (diagonal only, since columns are unique)
def count_attacks(state):
    n = len(state)
    attacks = 0
    for i in range(n):
        for j in range(i + 1, n):
            # Check diagonal attacks
            if abs(state[i] - state[j]) == abs(i - j):
                attacks += 1
    return attacks

# Fitness is the negative number of attacks (lower is better)
def fitness(state):
    return -count_attacks(state)

# Convert the board to a human-readable string
def board_to_string(state):
    n = len(state)
    lines = []
    for row in range(n):
        line = ""
        for col in range(n):
            line += "Q " if state[row] == col else ". "
        lines.append(line)
    return "\n".join(lines) + "\n"

# Print board visually
def print_board(state):
    print(board_to_string(state))

# Single-point crossover between two parents
def crossover(parent1, parent2):
    point = random.randint(1, len(parent1) - 2)
    child = parent1[:point] + parent2[point:]
    return child

# Mutate by randomly changing one queen's column
def mutate(individual, mutation_rate=0.7):
    new_individual = individual[:]
    if random.random() < mutation_rate:
        row = random.randint(0, len(individual) - 1)
        col = random.randint(0, len(individual) - 1)
        new_individual[row] = col
    return new_individual

# Tournament selection: pick k individuals and return the best
def select(population, k=3):
    selected = random.sample(population, k)
    selected.sort(key=lambda x: fitness(x), reverse=True)
    return selected[0]

# Main genetic algorithm for N-Queens
def genetic_algorithm(n, population_size=100, max_generations=1000):
    # Initialize population with permutations (no column conflict)
    population = [random.sample(range(n), n) for _ in range(population_size)]
    generation = 0

    while generation < max_generations:
        # Sort population by fitness (higher is better)
        population.sort(key=lambda x: fitness(x), reverse=True)
        best = population[0]

        # Found a solution with no attacks
        if count_attacks(best) == 0:
            return best, generation

        # Elitism: keep the best individual
        new_population = [best]

        # Generate new individuals via crossover and mutation
        while len(new_population) < population_size:
            parent1 = select(population)
            parent2 = select(population)
            child = crossover(parent1, parent2)
            child = mutate(child)
            new_population.append(child)

        population = new_population
        generation += 1

    # Return best found solution if max_generations reached
    population.sort(key=lambda x: fitness(x), reverse=True)
    return population[0], generation

# Run the algorithm multiple times and record results
def run_experiments(n, trials=30):
    success_count = 0
    total_time = 0.0
    output_lines = []

    output_lines.append(f"Running {trials} trials for {n}-Queens using Genetic Algorithm...\n")

    for i in range(trials):
        start_time = time.time()
        solution, gens = genetic_algorithm(n)
        elapsed_time = time.time() - start_time
        total_time += elapsed_time
        is_success = 1 if count_attacks(solution) == 0 else 0
        success_count += is_success

        output_lines.append(f"Trial {i+1:2d}: Time = {elapsed_time:.4f}s | Success = {is_success} | Generations = {gens}")
        output_lines.append("Final board for this trial:")
        output_lines.append(board_to_string(solution))

    success_rate = success_count / trials
    avg_time = total_time / trials

    output_lines.append("\n===== Final Report =====")
    output_lines.append(f"Success rate: {success_rate:.2%} ({success_count}/{trials})")
    output_lines.append(f"Average time: {avg_time:.4f} seconds")

    filename = f"{n}-Queens_GA_result.txt"
    with open(filename, "w") as f:
        f.write("\n".join(output_lines))

    print("\n".join(output_lines))


In [13]:
run_experiments(8, trials=30)

Running 30 trials for 8-Queens using Genetic Algorithm...

Trial  1: Time = 0.0090s | Success = 1 | Generations = 2
Final board for this trial:
. . . . . . Q . 
. . Q . . . . . 
Q . . . . . . . 
. . Q . . . . . 
Q . . . . . . . 
. . . . . . . Q 
. Q . . . . . . 
. . . . Q . . . 

Trial  2: Time = 0.0083s | Success = 1 | Generations = 2
Final board for this trial:
Q . . . . . . . 
. . . . Q . . . 
. Q . . . . . . 
. . . . . Q . . 
. . Q . . . . . 
. . . . . . Q . 
. . . Q . . . . 
. . . Q . . . . 

Trial  3: Time = 0.0054s | Success = 1 | Generations = 1
Final board for this trial:
. . . . Q . . . 
Q . . . . . . . 
. . . . . Q . . 
. . . . . Q . . 
. . . . . Q . . 
. Q . . . . . . 
. . . . Q . . . 
. . . . . . . Q 

Trial  4: Time = 0.0180s | Success = 1 | Generations = 4
Final board for this trial:
Q . . . . . . . 
. . . . . Q . . 
Q . . . . . . . 
Q . . . . . . . 
. . . . . Q . . 
. . . . . . . Q 
. Q . . . . . . 
. . . Q . . . . 

Trial  5: Time = 0.0060s | Success = 1 | Generations 

In [14]:
run_experiments(50, trials=30)

Running 30 trials for 50-Queens using Genetic Algorithm...

Trial  1: Time = 4.7008s | Success = 1 | Generations = 47
Final board for this trial:
. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . Q . . . . . . . . . . . . . . 
. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . Q . . . . . . . 
. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . Q . . . . . . . . . . . . . . . . . . 
. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . Q . . . . . . . . . . . . . . . 
. . . . . . . . . . . . . . . . . . . . . Q . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
. . . . . . . . . . . . . . . . . . Q . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
. . Q . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
. . . . . . . . . . . . . . . . . . . . . . . . . . . . . Q . . . . . . . . . . . . . . . . . . . . 
. . . . . . . . . . . . . . . . . . . . Q . . 